# 🌿 Análise Estatística — Base de Dados da Emergy Society
**Comparação entre UEV com e sem Trabalho e Serviços (L&S)**

---

| Seção | Conteúdo |
|---|---|
| **0. Setup** | Upload do arquivo Excel + instalação de dependências |
| **1. Dados** | Carregamento, filtragem e primeiras linhas |
| **2. Estatística Descritiva** | N, mínimo, máximo, média, mediana, desvio padrão |
| **3. Razão k** | k = UEV_with / UEV_without — análise e gráficos |
| **4. Correlação de Pearson** | Coeficiente r, p-valor, dispersão, matriz |
| **5. Regressão Linear** | y = a + b·x — coeficientes, R², resíduos |
| **6. Regressão Log-Log** | log(y) = α + β·log(x) — lei de potência, diagnóstico |

---
> **Como usar:** Execute as células **em ordem**, de cima para baixo.  
> Comece pela **Seção 0** — ela faz o upload do arquivo Excel que todas as outras seções precisam.

---
## 0. ⚙️ Setup — Upload do Arquivo e Dependências

**Execute esta célula primeiro.** Ela vai:
1. Instalar o `scipy` (necessário para correlação e regressão)
2. Abrir um botão para você fazer upload do `EmergyDatabase.xlsx`

In [ ]:
# Instalar dependência
!pip install scipy --quiet

# Upload do arquivo Excel
from google.colab import files

print('📂 Clique em "Escolher arquivos" e selecione o EmergyDatabase.xlsx')
uploaded = files.upload()

print('\n✅ Arquivos recebidos:')
for name in uploaded.keys():
    print(f'   → {name}')

---
## 1. 📋 Carregamento e Filtragem dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuração global dos gráficos
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Carregar planilha
df = pd.read_excel('EmergyDatabase.xlsx')

print(f'📊 Dimensões originais : {df.shape[0]} linhas × {df.shape[1]} colunas')
print(f'📋 Colunas disponíveis :')
for col in df.columns:
    print(f'   • {col}')

In [ ]:
col_with    = 'Unit Emergy Value (UEV) with Labor and Services'
col_without = 'Unit Emergy Value (UEV) without Labor and Services'

# Converter para numérico — valores não numéricos viram NaN
df['y'] = pd.to_numeric(df[col_with],    errors='coerce')
df['x'] = pd.to_numeric(df[col_without], errors='coerce')

# Manter apenas pares com ambos os valores numéricos e positivos
clean = (
    df.dropna(subset=['x', 'y'])
      .loc[lambda d: (d['x'] > 0) & (d['y'] > 0)]
      .copy()
      .reset_index(drop=True)
)

print(f'📌 Registros originais  : {len(df)}')
print(f'✅ Pares válidos        : {len(clean)}')
print(f'❌ Registros excluídos  : {len(df) - len(clean)}')
print()
print('📏 Distribuição de unidades nos pares válidos:')
print(clean['Units'].value_counts().to_string())

In [ ]:
# Primeiras 10 linhas do conjunto filtrado
print('🔍 Primeiras 10 linhas do conjunto final:')
clean[['Title', 'x', 'y', 'Units']].head(10).style \
    .format({'x': '{:.3e}', 'y': '{:.3e}'}) \
    .set_caption('Conjunto de dados filtrado (primeiras 10 linhas)') \
    .background_gradient(subset=['x','y'], cmap='Blues')

---
## 2. 📊 Estatística Descritiva

In [ ]:
desc = pd.DataFrame({
    'UEV com L&S  (y)': {
        'N'            : len(clean),
        'Mínimo'       : clean['y'].min(),
        'Máximo'       : clean['y'].max(),
        'Média'        : clean['y'].mean(),
        'Mediana'      : clean['y'].median(),
        'Desvio Padrão': clean['y'].std(),
    },
    'UEV sem L&S  (x)': {
        'N'            : len(clean),
        'Mínimo'       : clean['x'].min(),
        'Máximo'       : clean['x'].max(),
        'Média'        : clean['x'].mean(),
        'Mediana'      : clean['x'].median(),
        'Desvio Padrão': clean['x'].std(),
    },
})

desc.style \
    .format('{:.4e}', subset=pd.IndexSlice[['Mínimo','Máximo','Média','Mediana','Desvio Padrão'], :]) \
    .set_caption('Estatística Descritiva — UEV com e sem L&S') \
    .set_properties(**{'text-align': 'center'})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Boxplot ──────────────────────────────────────────────────────────────────
ax = axes[0]
bp = ax.boxplot(
    [np.log10(clean['y']), np.log10(clean['x'])],
    labels=['UEV com L&S (y)', 'UEV sem L&S (x)'],
    patch_artist=True, widths=0.45,
)
for patch, color in zip(bp['boxes'], ['#4c9be8', '#f28c38']):
    patch.set_facecolor(color); patch.set_alpha(0.75)
ax.set_ylabel('log₁₀(UEV)  [seJ]', fontsize=12)
ax.set_title('Boxplot comparativo (escala log)', fontsize=13, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)

# ── Histogramas sobrepostos ───────────────────────────────────────────────────
ax2 = axes[1]
for col, label, color in [('y','UEV com L&S','#4c9be8'), ('x','UEV sem L&S','#f28c38')]:
    ax2.hist(np.log10(clean[col]), bins=15, color=color, edgecolor='white',
             alpha=0.6, label=label)
    ax2.axvline(np.log10(clean[col].median()), linestyle='--', lw=1.8,
                color=color, label=f'Mediana {label} = {clean[col].median():.2e}')
ax2.set_xlabel('log₁₀(UEV)', fontsize=12)
ax2.set_ylabel('Frequência', fontsize=12)
ax2.set_title('Histograma comparativo (escala log)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Distribuição dos valores de UEV', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. 📐 Razão k = UEV_with / UEV_without

| Situação | Interpretação |
|---|---|
| **k > 1** | L&S eleva o UEV — serviços agregam emergia |
| **k ≈ 1** | L&S tem impacto desprezível |
| **k < 1** | L&S reduz o UEV — possível inconsistência metodológica |

In [ ]:
clean['k'] = clean['y'] / clean['x']
k = clean['k']

print('─' * 42)
print(f'  Mínimo   k : {k.min():.4f}')
print(f'  Máximo   k : {k.max():.4f}')
print(f'  Mediana  k : {k.median():.4f}')
print(f'  Média    k : {k.mean():.4f}')
print('─' * 42)
print(f'  k > 1      : {(k > 1).sum():>3} pares  ({(k > 1).mean()*100:.1f}%)')
print(f'  k ≈ 1 (±1%): {((k>=0.99)&(k<=1.01)).sum():>3} pares')
print(f'  k < 1      : {(k < 1).sum():>3} pares  ({(k < 1).mean()*100:.1f}%)')
print('─' * 42)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histograma de k (log) ────────────────────────────────────────────────────
ax = axes[0]
ax.hist(np.log10(k), bins=20, color='#4c9be8', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', lw=2, label='k = 1  (log = 0)')
ax.axvline(np.log10(k.median()), color='navy', linestyle='-.', lw=1.8,
           label=f'Mediana k = {k.median():.3f}')
ax.set_xlabel('log₁₀(k)', fontsize=12)
ax.set_ylabel('Frequência', fontsize=12)
ax.set_title('Distribuição da razão k (escala log)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# ── Scatter UEV_with vs UEV_without colorido por k ───────────────────────────
ax2 = axes[1]
sc = ax2.scatter(
    np.log10(clean['x']), np.log10(clean['y']),
    c=np.log10(k), cmap='RdYlGn', s=70, alpha=0.85,
    edgecolors='gray', linewidths=0.4
)
lim = [min(np.log10(clean['x']).min(), np.log10(clean['y']).min()) - 0.3,
       max(np.log10(clean['x']).max(), np.log10(clean['y']).max()) + 0.3]
ax2.plot(lim, lim, 'k--', lw=1.5, alpha=0.5, label='k = 1 (y = x)')
plt.colorbar(sc, ax=ax2, label='log₁₀(k)')
ax2.set_xlabel('log₁₀(UEV sem L&S)', fontsize=12)
ax2.set_ylabel('log₁₀(UEV com L&S)', fontsize=12)
ax2.set_title('Dispersão colorida pela razão k', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(linestyle='--', alpha=0.4)

plt.suptitle('Análise da Razão k', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
cols_show = ['Title', 'Units', 'x', 'y', 'k']

print('═══ Top 10 maiores k ═══')
display(
    clean[cols_show].nlargest(10, 'k').reset_index(drop=True)
    .style.format({'x': '{:.3e}', 'y': '{:.3e}', 'k': '{:.4f}'})
    .background_gradient(subset=['k'], cmap='Greens')
)

print('\n═══ Top 10 menores k ═══')
display(
    clean[cols_show].nsmallest(10, 'k').reset_index(drop=True)
    .style.format({'x': '{:.3e}', 'y': '{:.3e}', 'k': '{:.4f}'})
    .background_gradient(subset=['k'], cmap='Reds_r')
)

---
## 4. 🔗 Correlação de Pearson

| Intervalo de \|r\| | Intensidade |
|---|---|
| 0,00 – 0,19 | Muito fraca |
| 0,20 – 0,39 | Fraca |
| 0,40 – 0,59 | Moderada |
| 0,60 – 0,79 | Forte |
| 0,80 – 1,00 | Muito forte |

In [ ]:
r, p_value = stats.pearsonr(clean['x'], clean['y'])

forca = 'muito forte' if abs(r) >= 0.80 else \
        'forte'       if abs(r) >= 0.60 else \
        'moderada'    if abs(r) >= 0.40 else 'fraca'
sinal = 'positiva' if r > 0 else 'negativa'

print('─' * 42)
print(f'  Coeficiente r  : {r:.4f}')
print(f'  p-valor        : {p_value:.2e}')
print(f'  Significativo? : {"Sim" if p_value < 0.05 else "Não"}  (α = 0,05)')
print('─' * 42)
print(f'\n→ Correlação {sinal} e {forca} (r = {r:.4f})')

In [ ]:
log_x = np.log10(clean['x'])
log_y = np.log10(clean['y'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Dispersão log-log com linha de identidade ─────────────────────────────────
ax = axes[0]
ax.scatter(log_x, log_y, alpha=0.65, color='#4c9be8',
           edgecolors='#1a4a7a', s=60, label=f'Pares válidos (n={len(clean)})')
lim_min = min(log_x.min(), log_y.min()) - 0.2
lim_max = max(log_x.max(), log_y.max()) + 0.2
ax.plot([lim_min, lim_max], [lim_min, lim_max],
        color='red', linestyle='--', lw=1.5, alpha=0.7, label='y = x (referência)')
ax.set_xlim(lim_min, lim_max); ax.set_ylim(lim_min, lim_max)
ax.set_xlabel('log₁₀(UEV sem L&S)  →  x', fontsize=12)
ax.set_ylabel('log₁₀(UEV com L&S)  →  y', fontsize=12)
ax.set_title(f'Dispersão log-log\nPearson: r = {r:.4f}  (p = {p_value:.1e})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(linestyle='--', alpha=0.4); ax.set_aspect('equal')

# ── Matriz de correlação ──────────────────────────────────────────────────────
ax2 = axes[1]
corr_matrix = clean[['x','y']].rename(columns={'x':'UEV_without','y':'UEV_with'}).corr()
im = ax2.imshow(corr_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax2)
ax2.set_xticks([0,1]); ax2.set_xticklabels(['UEV_without','UEV_with'], fontsize=11)
ax2.set_yticks([0,1]); ax2.set_yticklabels(['UEV_without','UEV_with'], fontsize=11)
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f'{corr_matrix.iloc[i,j]:.4f}',
                 ha='center', va='center', fontsize=14, fontweight='bold',
                 color='white' if corr_matrix.iloc[i,j] > 0.6 else 'black')
ax2.set_title('Matriz de Correlação de Pearson', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 5. 📈 Regressão Linear — Escala Original

> **Modelo:** y = a + b · x  
> onde x = UEV_without  e  y = UEV_with

In [ ]:
slope, intercept, r_lin, p_lin, std_err = stats.linregress(clean['x'], clean['y'])
R2_lin = r_lin ** 2

print('─' * 48)
print(f'  Intercepto  (a)  : {intercept:.4e}')
print(f'  Coef. angular (b): {slope:.6f}')
print(f'  R²               : {R2_lin:.4f}')
print(f'  p-valor          : {p_lin:.2e}')
print(f'  Erro padrão      : {std_err:.4e}')
print('─' * 48)
print(f'\n  Equação: y = {intercept:.4e} + {slope:.6f} · x')

In [ ]:
y_pred   = intercept + slope * clean['x']
residuos = clean['y'] - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Dispersão + reta ajustada ─────────────────────────────────────────────────
ax = axes[0]
ax.scatter(clean['x'], clean['y'], alpha=0.65, color='#4c9be8',
           edgecolors='#1a4a7a', s=65, zorder=3, label=f'n = {len(clean)}')
x_line = np.linspace(clean['x'].min(), clean['x'].max(), 400)
ax.plot(x_line, intercept + slope * x_line, color='#e84c4c', lw=2.2, zorder=4,
        label=f'y = {intercept:.2e} + {slope:.4f}·x\nR² = {R2_lin:.4f}')
ax.set_xlabel('UEV sem L&S  (x)  [seJ]', fontsize=11)
ax.set_ylabel('UEV com L&S  (y)  [seJ]', fontsize=11)
ax.set_title('Regressão Linear\nEscala Original', fontsize=12, fontweight='bold')
ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))
ax.legend(fontsize=9); ax.grid(linestyle='--', alpha=0.4)

# ── Resíduos vs Ajustados ────────────────────────────────────────────────────
ax2 = axes[1]
ax2.scatter(y_pred, residuos, alpha=0.65, color='#f28c38',
            edgecolors='#8c4a00', s=55)
ax2.axhline(0, color='red', linestyle='--', lw=1.5)
ax2.set_xlabel('Valores ajustados  ŷ', fontsize=11)
ax2.set_ylabel('Resíduo  (y − ŷ)', fontsize=11)
ax2.set_title('Resíduos vs. Valores Ajustados', fontsize=12, fontweight='bold')
ax2.ticklabel_format(style='sci', axis='both', scilimits=(0,0))
ax2.grid(linestyle='--', alpha=0.4)

# ── Histograma dos resíduos ───────────────────────────────────────────────────
ax3 = axes[2]
ax3.hist(residuos, bins=15, color='#4c9be8', edgecolor='white', alpha=0.85)
ax3.set_xlabel('Resíduo', fontsize=11)
ax3.set_ylabel('Frequência', fontsize=11)
ax3.set_title('Distribuição dos Resíduos', fontsize=12, fontweight='bold')
ax3.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
ax3.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Diagnóstico — Regressão Linear', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Análise de sensibilidade — remoção dos 5% extremos
q_low  = clean['x'].quantile(0.05)
q_high = clean['x'].quantile(0.95)
trimmed = clean[(clean['x'] >= q_low) & (clean['x'] <= q_high)]

slope_t, intercept_t, r_t, _, _ = stats.linregress(trimmed['x'], trimmed['y'])

print(f'Dataset aparado (5%–95%): n = {len(trimmed)}')
print(f'  Intercepto (a) : {intercept_t:.4e}')
print(f'  Coef. ang. (b) : {slope_t:.6f}')
print(f'  R²             : {r_t**2:.4f}')
print(f'  Δ R²           : {r_t**2 - R2_lin:+.4f}  vs. modelo completo')

---
## 6. 📉 Regressão Log-Log

> **Modelo:** log₁₀(y) = α + β · log₁₀(x)  
> **Equivalente:** y = 10^α · x^β  (lei de potência)

> Este modelo é preferível quando os dados variam por várias ordens de grandeza.

In [ ]:
log_x = np.log10(clean['x'])
log_y = np.log10(clean['y'])

beta, alpha, r_log, p_log, std_log = stats.linregress(log_x, log_y)
R2_log = r_log ** 2

print('─' * 52)
print(f'  α (intercepto log) : {alpha:.4f}')
print(f'  β (expoente)       : {beta:.4f}')
print(f'  R²                 : {R2_log:.4f}')
print(f'  p-valor            : {p_log:.2e}')
print(f'  Erro padrão        : {std_log:.4f}')
print('─' * 52)
print(f'\n  Equação log-log : log(y) = {alpha:.4f} + {beta:.4f} · log(x)')
print(f'  Lei de potência : y = {10**alpha:.4e} · x^{beta:.4f}')

if abs(beta - 1) < 0.05:
    print(f'\n  → β ≈ 1: relação quase proporcional (lei de potência quase linear)')
elif beta > 1:
    print(f'\n  → β > 1: UEV_with cresce mais rápido que UEV_without')
else:
    print(f'\n  → β < 1: UEV_with cresce mais lentamente que UEV_without')

In [ ]:
C = 10 ** alpha
log_y_pred = alpha + beta * log_x
res_log    = log_y - log_y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Gráfico log-log com reta ajustada ─────────────────────────────────────────
ax = axes[0]
ax.scatter(log_x, log_y, alpha=0.65, color='#f28c38',
           edgecolors='#8c4a00', s=65, zorder=3, label=f'n = {len(clean)}')
x_fit = np.linspace(log_x.min(), log_x.max(), 400)
ax.plot(x_fit, alpha + beta * x_fit, color='#e84c4c', lw=2.2, zorder=4,
        label=f'log(y) = {alpha:.4f} + {beta:.4f}·log(x)\nR² = {R2_log:.4f}')
lim = [min(log_x.min(), log_y.min())-0.3, max(log_x.max(), log_y.max())+0.3]
ax.plot(lim, lim, color='gray', linestyle=':', lw=1.5, alpha=0.7, label='referência (β=1)')
ax.set_xlabel('log₁₀(UEV sem L&S)', fontsize=11)
ax.set_ylabel('log₁₀(UEV com L&S)', fontsize=11)
ax.set_title('Regressão Log-Log', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(linestyle='--', alpha=0.4)

# ── Lei de potência na escala original ───────────────────────────────────────
ax2 = axes[1]
ax2.scatter(clean['x'], clean['y'], alpha=0.65, color='#f28c38',
            edgecolors='#8c4a00', s=65, zorder=3, label=f'n = {len(clean)}')
x_orig = np.linspace(clean['x'].min(), clean['x'].max(), 400)
ax2.plot(x_orig, C * x_orig**beta, color='#e84c4c', lw=2.2, zorder=4,
         label=f'y = {C:.3e} · x^{beta:.4f}\nR² = {R2_log:.4f}')
ax2.set_xlabel('UEV sem L&S  (x)  [seJ]', fontsize=11)
ax2.set_ylabel('UEV com L&S  (y)  [seJ]', fontsize=11)
ax2.set_title('Lei de Potência\nEscala Original', fontsize=12, fontweight='bold')
ax2.ticklabel_format(style='sci', axis='both', scilimits=(0,0))
ax2.legend(fontsize=9); ax2.grid(linestyle='--', alpha=0.4)

# ── Resíduos log ─────────────────────────────────────────────────────────────
ax3 = axes[2]
ax3.scatter(log_y_pred, res_log, alpha=0.65, color='#4c9be8',
            edgecolors='#1a4a7a', s=55)
ax3.axhline(0, color='red', linestyle='--', lw=1.5)
ax3.set_xlabel('log(ŷ) — valores ajustados', fontsize=11)
ax3.set_ylabel('Resíduo log  (log y − log ŷ)', fontsize=11)
ax3.set_title('Resíduos vs. Ajustados (log)', fontsize=12, fontweight='bold')
ax3.grid(linestyle='--', alpha=0.4)

plt.suptitle('Diagnóstico — Regressão Log-Log', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. 🏁 Comparativo Final — Linear vs. Log-Log

In [ ]:
comp = pd.DataFrame({
    'Modelo'              : ['Linear (escala original)', 'Log-Log (lei de potência)'],
    'Equação'             : [f'y = {intercept:.2e} + {slope:.4f}·x',
                             f'log(y) = {alpha:.4f} + {beta:.4f}·log(x)'],
    'R²'                  : [round(R2_lin, 4), round(R2_log, 4)],
    'Escala'              : ['Original', 'Logarítmica'],
    'Sensível a outliers' : ['Alta', 'Baixa'],
    'Recomendado para UEV': ['Não', '✅ Sim'],
})

comp.style \
    .set_caption('Comparativo dos Modelos de Regressão') \
    .background_gradient(subset=['R²'], cmap='RdYlGn') \
    .set_properties(**{'text-align': 'center'})